# Working with Simulation Output


In this tutorial, we will take a closer look at what is in our simulation output. We will build upon our previous tutorial on using uproot and build the first steps of something we could use to analyse our data. Before we begin though, we will take a look at plotting our data as histograms, so that we can better understand what is going on. Of course, we now also have the benefit of a but of background information on what is actually going on in our detector and simulation. So now, we can begin to comprehend and interpret what our plots will show.

# Setup

Run the cells below once before running subsequent sections. The first one may take a minute or so. To summarise each cell -

- We are installing some packages (if they aren't already installed).
- Importing some packages.
- Setting a bunch of configuration options for our plots to make them look a bit nicer.

Once done, I recommend minimising this section.

In [ ]:
!pip install vector

In [ ]:
#Import some packages we'll need, specifically, uproot
import uproot as up
import os
import awkward as ak
import numpy as np
import pandas as pd
import scipy
import matplotlib as mpl
import matplotlib.ticker as ticker
import matplotlib.cm as cm
import matplotlib.pylab as plt
import vector
from XRootD import client
from scipy import stats
from matplotlib import pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib import colors as colours

In [ ]:
plt.rcParams['figure.figsize'] = [8.0, 6.0]
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xaxis.labellocation'] = 'right'
plt.rcParams['yaxis.labellocation'] = 'top'
SMALL_SIZE = 10
MEDIUM_SIZE = 14
BIGGER_SIZE = 20
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=MEDIUM_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=MEDIUM_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title]
deg2rad = np.pi/180.0

# Basic Plotting Introduction

We will begin by taking a look at some of our reconstructed charged particles again.

In [ ]:
 # The file we downloaded previously, change as desired
fname = "/home/jovyan/eic/Day_1_Tutorial_Input.root"
if os.path.isfile(fname):
    file=up.open(fname)
else:
    print("Error opening file - ", fname, " check your fname variable!")

In [ ]:
tree = file['events']
ReconChPartBr = tree["ReconstructedChargedParticles"].arrays()

So long as we had no errors above, we should now have our file opened and our events tree assigned. We have then converted our ReconstructedChargedParticles branch elements to an array.

We can now go ahead and plot these as a histogram (relatively) straightforwardly.

**Note** - Feel free to rename variables as you like, I've just assigned these things names that make sense to *me*. This doesn't neccessarily mean they will make sense to you :)

In [ ]:
# Optional - remind yourself of the branches we have available by uncommeting the for loop below and running it
#for entry in ReconChPartBr.fields:
#    print(entry)
# Note that at this point, our ReconChPartBr variable is an AwkwardArray .fields provides a list of our branch names

## Making a Basic Histogram

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"]))

As histograms go, not that informative. So what's gone wrong?

## Quick Quiz

How can we get a better looking plot?

## Better Looking Histograms

To get a slightly more useful plot, let's try to set the number of bins (and binning range) ourselves. We can also supress that print out of numbers before the plot by adding "plt.show()"

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"]), bins=100, range=(0,10))
plt.show()

Ok, better, but are we missing anything? Our plot before had a range going up to 1500.

This wasn't decided randomly, matplotlib tried to choose a range which would include all of our data. Maybe we should check some information regarding our array before deciding upon the binning.

In [ ]:
print("The smallest value in our array is:",np.min(ReconChPartBr["ReconstructedChargedParticles.energy"]))
print("The largest value in our array is:",np.max(ReconChPartBr["ReconstructedChargedParticles.energy"]))
print("The mean value of our array is:",np.mean(ReconChPartBr["ReconstructedChargedParticles.energy"]))
print("The standard deviation of values in our array is:",np.std(ReconChPartBr["ReconstructedChargedParticles.energy"]))

## Quick Quiz

Based upon these numbers, what might be a sensible range for our histogram?

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"]), bins=600, range=(0,300))
plt.show()

Much better, but we're not actually using much of our range, we see most events between 0 and 25, with some slight fuzz above that. Let's reduce it to 0 to 50 (stil in 0.5 GeV bins)

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"]), bins=100, range=(0,50))
plt.show()

## Combining What We Know - Uproot Selection Masks/Cuts

So we can turn our arrays into histograms. Useful, but it still doesn't tell us that much. However, we can really start cooking when we combine this with what we ended with in our uproot session. We can apply selection cuts to our arrays to fiter out events we don't want. **Importantly**, we can apply cuts on **other** quantities as we draw the one we want. Let's use this to check the energy of our negatively charged reconstructed tracks.


In [ ]:
Positive = ReconChPartBr['ReconstructedChargedParticles.charge'] > 0
Negative = ReconChPartBr['ReconstructedChargedParticles.charge'] < 0
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50))
plt.show()

Nice, but how do our negative and positive charged particle energies compare? Well, we could plot them on top of each other and see -

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50),alpha=0.5)
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Positive]), bins=100, range=(0,50),alpha=0.5,color='r')
plt.show()

We've added some extra options to our histograms here such that each plot has some transparency *and* the positively chagred particles are drawn in red rather than blue.

We could also have created the plot above slightly differently - 

In [ ]:
plt.hist([ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), 
          ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Positive])],
         bins=100, range=(0, 50),color=['b','r'])
# Note the square bracket enclosing our arrays and the colour selection we've chosen.
plt.show()

There are a few other tweaks we may wish to consider making to our histogram too.

### Histogram Plotting Options

There's a lot more we can (and probably should) do to our histogram to make it a bit nicer looking. To begin with, axis labels -

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50),alpha=0.5)
plt.xlabel('E [GeV]')
plt.ylabel('# Entries / 0.5 GeV')
plt.show()

If we want to use latex characters, we can enclose sections in dollar symbols too. For example - $P_{x}$ (double click this section to see the raw text input).

We can also add a title to our plot and set the x/y axis to be displayed logarithmically.

In this particular caplt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50),alpha=0.5)
plt.xlabel('E [GeV]')
plt.ylabel('# Entries / 0.5 GeV')
plt.show()se, that's maybe not so useful but, it's a option so let's try it -

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50),alpha=0.5)
plt.xlabel('E [GeV]')
plt.ylabel('# Entries / 0.5 GeV')
plt.title("Energy of Negatively Charged Reconstructed Particles")
plt.xscale('log')
plt.yscale('log')
plt.show()

Not so useful, so we'll not do that again at the moment.

For our plot of positively and negatively charged particles on the same plot, we should also add a legend though so we know what we're looking at -

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50),alpha=0.5, label="-ve Particles")
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Positive]), bins=100, range=(0,50),alpha=0.5,color='r', label="+ve Particles")
plt.xlabel('E [GeV]')
plt.ylabel('# Entries / 0.5 GeV')
plt.title("Energy of Charged Reconstructed Particles")
plt.legend(loc='upper right')
plt.show()

Finally, we might want to save our histogram to a file.

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50),alpha=0.5, label="-ve Particles")
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Positive]), bins=100, range=(0,50),alpha=0.5,color='r', label="+ve Particles")
plt.xlabel('E [GeV]')
plt.ylabel('# Entries / 0.5 GeV')
plt.title("Energy of Charged Reconstructed Particles")
plt.legend(loc='upper right')
plt.savefig("MyFirstHistogram.png") # Note - the ordering matters here, if we do this AFTER showing the plot, the file will be blank
plt.show()

## Exercise

So far, we have looked at our ReconstructedChargedParticles branch. Another very important branch that we will make extensive use of is the "MCParticles" branch. This contains the "truth" infromation for our events. I.e. it tells us what we *actually* fed in to our simulation. In this exercise, we will take a look at some of this MC information and compare it (simplistically) to our reconstructed information. To begin with -

In [ ]:
MCPartBr = tree["MCParticles"].arrays()
ElecMass = 511*(10**-6) # Electron mass in GeV

You may wish to uncomment the cell below and run it to check what branch elements are contained within the MCParticles branch -

In [ ]:
#tree["MCParticles"].keys()

Your task is to -

1. Identify scattered electrons in the MCParticles branch, plot the components of their momentum and their energy.
2. Make plots that compare these quantities to the reconstructed charged particles that we believe are the scattered electrons.

*Hint* - For *stable*, charged particles (non-intermediate states in our simulation), the generator status should be equal to 1. Also, the PDG code for an electron is 11.
Note that for the energy, we will need to determine this quantity for our MC electrons.

## Comments

Our work on the reconstrucred charged particles branch was actually a slightly sneaky trap to lead us into the next discussion. We can utilise some of the the tools and techniques we've just developed to start a rudimentary analysis. *However*, we will need to be a bit more careful with how we handle our reconstructed particles. In particular, the issue is that our PDG value is not (currently) as reliable as we might like. We will instead utilise *associations*.

# Basic Analysis

In this section, we will make some basic analysis plots checking the detection efficiency and resolution for some of our particles. Before that though, we will take a look at associations and how we can utilise them.

## Associations

*Associations* are a mapping between our input MC particles and the tracks we reconstruct. Without going into too many details of the simulation, associations effectively allow us to *check* if a reconstructed track that we see actually corresponds to an MC particle that was generated. This allows us to check that the reconstructed track information we're plotting actually comes from the reaction we're interested in.

Of course, this isn't something we can *actually* do in reality. We don't *know* what the "truth" is for real events. However, in the absence of full PID (for now), we can make use of these associations. We can also use them to check some basic quantities such as our detetor efficiency and resolution (as we will discuss soon). Using associations is a matter of matching indices between various arrays -

In [ ]:
RecoAssoc = tree['ReconstructedChargedParticleAssociations'].arrays()
tree['ReconstructedChargedParticleAssociations'].keys()

The two quantities we will be interested in are simID and recID. We need to check if the index of the particle we're interested in from our MC branch matches one of the SimID values. We then take the index of the value that matches and get the reconstructed branch element with this value. So for example -

- Electron in our MCParticles branch is the *third* entry
    - Check to see if the entries in ReconstructedChargedParticleAssociations.simID match 3 for this event
        - The 2nd entry of simID for this event matches, therefore, take the 2nd value of recID for this event and get *this* entry index from the reconstructed charged particles branch for this event 

In code, we could implement this as below -

In [ ]:
RecID=RecoAssoc['ReconstructedChargedParticleAssociations.recID'] # Array of reconstructed IDs
SimID=RecoAssoc['ReconstructedChargedParticleAssociations.simID'] # Array of simulated IDs
BoolMatch=(MCPartBr["MCParticles.PDG"][SimID])==(ReconChPartBr["ReconstructedChargedParticles.PDG"][RecID]) # Use simulated or reconstructed IDs as indices, this checks if the pdg between each array matches
BoolChargeTrack = ((abs(MCPartBr["MCParticles.charge"][SimID])!=0) & (MCPartBr["MCParticles.generatorStatus"][SimID]==1))
BoolElec=((MCPartBr["MCParticles.PDG"][SimID]==11) & (MCPartBr["MCParticles.generatorStatus"][SimID]==1))
BoolPion=((abs(MCPartBr["MCParticles.PDG"][SimID])==211) & (MCPartBr["MCParticles.generatorStatus"][SimID]==1)) # Use abs to include both positive and negative pions

# We also require that these are stable thrown particles

print("Number of thrown particles with matching track: ",np.asarray(ak.flatten(BoolChargeTrack)).sum())
print("Number of thrown electrons with matching track: ",np.asarray(ak.flatten(BoolElec)).sum())
print("Number of thrown pions with matching track: ",np.asarray(ak.flatten(BoolPion)).sum())

We can use these booleans to select out cases where we successfully detect particles of interest -

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][RecID][BoolElec]), bins=100, range=(0,25),alpha=0.5, color='r', label="Rec")
plt.xlabel('E [GeV]')
plt.ylabel('# Entries / 0.25 GeV/c')
plt.title("$e'$ E")
plt.show()

## Exercise

Remake your plots for part 2 of the previous exercise using, drawing only events with matching assoiations.

*Hint* - You may want (or need) to recalculate the MC energies.

## Efficiency and Resolution

In this section, we will generate some basic physics analysis plots. Specifically, we will look at *efficiency* and *resolution*. Before we get to that though, we're going to need to do one other thing with our data.

So far, we've been looking at quantities individually, e.g. just the energy and so on. However, it is often more convnenient (and useful) to think of our produced particles in terms of their vectors. As such, we will create and assign 3/4-vectors for our MC and reconstructed particles.

### Vectors

In [ ]:
MC_Parts = vector.zip({'px': MCPartBr["MCParticles.momentum.x"], 'py': MCPartBr["MCParticles.momentum.y"], 'pz': MCPartBr["MCParticles.momentum.z"]})
Rec_Parts = vector.zip({'px': ReconChPartBr["ReconstructedChargedParticles.momentum.x"], 'py': ReconChPartBr["ReconstructedChargedParticles.momentum.y"], 'pz': ReconChPartBr["ReconstructedChargedParticles.momentum.z"], 'E':ReconChPartBr["ReconstructedChargedParticles.energy"]})

In [ ]:
print(MC_Parts[1])
print(Rec_Parts[1])

Creating 3 or 4-vectors in this way is useful as we can then use various functions to extract information from our vectors. For example, we can easily get -

- *Pseudorapidity*, $\eta$
- *Polar angle*, $\theta$ they make wrt the origin (in the lab frame, our bunch crossing point) - *Note, this is in radians by default*
- *Transverse momentum*, $P_{T}$
- ...

In [ ]:
print(Rec_Parts.eta)
print(Rec_Parts.theta)
print(Rec_Parts.pt)
print(Rec_Parts.p)

We can of course, also apply selection cuts to our vector arrays as we did before too. So, we could plot our $e'$ pseudorapidity for our reconstructed events for example via -

In [ ]:
plt.hist(ak.flatten((Rec_Parts[RecID][BoolElec]).eta), bins=100, range=(-5,5),alpha=0.5, color='g')
plt.xlabel('$\eta$')
plt.ylabel('# Entries')
plt.title("Reconstructed $e'$ $\eta$")
plt.show()

#### Quick Quiz

Does our distribution above make sense? Where would we typically expect $e'$ to go?

### Efficiency

Efficiency is a measure of the probability that we will detect an incident particle. We could calculate our efficiency by straightforwardly counting how many particles we detect vs how many we "threw". For example, if we detect 9 particles and 10 were generated, our efficiency is -

9/10

i.e.

90%

This might be a useful figure. However, if we are evaluating the performance of a detector, it might be useful to consider the efficiency as a function of another quantity, for example. $\eta$ or $P$. What this will tell us is how likely we are to detect particles incident on certain areas of the detector (or with a certain momentum for instance). *We should not really expect these distributions to be completely flat*. 

In terms of our code, we can straightforwardly determine this by dividing some histograms. We can divide histograms via -

In [ ]:
MCHist = np.histogram(ak.flatten(MCEnerElec), bins=100, range=(0,25))
RecHist = np.histogram(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][RecID][BoolElec]), bins=100, range=(0,25))
with np.errstate(divide='ignore'):
    Division = RecHist[0] / MCHist[0]
Division = np.nan_to_num(Division,nan=0, posinf = 0)
Bin_Edges=MCHist[1]
Bars = 0.5 * (Bin_Edges[1:] + Bin_Edges[:-1])
BarWidth=Bars[1]-Bars[0]
plt.bar(Bars, Division, width=BarWidth, alpha=0.5, color='g')

Here we have used the numpy histogram function to turn our arrays into a histogram. MC or Rec Hist is an array where our 0th set is the bin contents and the 1st set is the edges of our bins. Somehwat tediously, then we need to manipulate this to get the bin centres and bin width. We then make a new bar chart using this (rather than a histogram as we don't need to manupulate the array any further).

As an aside, this somewhat tedious process is a slight downside of uproot/python. If we used ROOT directly, we could just call Hist1->Divide(Hist2).

**Important!** - What we've just created is not our efficiency, we'll be doing that in the exercise below!

### Resolution

Another very useful thing we may want to calculate is the resolution of a particular quantity. The resolution tells us how well we can reconstruct our "true" value. For example. we might want to know how well we can determine the energy of our particles. As such, we could calculate the energy resolution. Our resolution is simply -

- (Reconstructed - True)/True

This is often expressed as a percentage. So, for example, say we detect a particle and determine its energy to be 0.95 GeV. In reality, the energy was 1 GeV. As such, our energy resolution for this particle is - 

- 0.95-1/1 = -5%

Let's see a quick example of this calculation for a a quantity -

In [ ]:
ElecMomXRes = ((ReconChPartBr["ReconstructedChargedParticles.momentum.x"][RecID][BoolElec]-MCPartBr["MCParticles.momentum.x"][SimID][BoolElec])/MCPartBr["MCParticles.momentum.x"][SimID][BoolElec])*100
plt.hist(ak.flatten(ElecMomXRes), bins=50, range=(-25,25),alpha=0.5, color='g')
plt.xlabel("$e'$ $P_{x}$ Resolution [%]")
plt.ylabel('# Entries')
plt.title("Reconstructed $e'$ $P_{x}$ Resolution")
plt.show()

#### 2D Histograms

2D histograms can be a very useful tool to spot correlations in our data. For example, do we see a drop off in $P_{x}$ resolution as a function of track momentum? We can easily check this with a 2D histogram where our x-axis is our $P_{x}$ resolution and the y-axis is the true momentum of the track, $P_{MC}$.

In [ ]:
plt.hist2d(np.asarray(ak.flatten(ElecMomXRes)), np.asarray(ak.flatten(MC_Parts[SimID][BoolElec].p)), bins=[50,50], range=[[-100,100],[0,20]], cmin=1)
plt.xlabel("$e'$ $P_{x}$ Resolution [%]")
plt.ylabel("$e'$ $P_{MC}$")
plt.title("Reconstructed $e'$ $P_{x}$ Resolution as a function of True Momentum")
cb = plt.colorbar()
cb.set_label('Counts/bin')
plt.show()

**Note** - Matplotlib doesn't like dealing with awkard arrays, so we had to convert these to "normal" numpy arrays here. Another fun bit of python jank.

### Exercise

1. Determine the efficiency as a function of $\eta$ and $P$ for $e'$ and pions in our file.
2. Determine the momentum resolution, P, and $\eta$ resolution of $e'$ and pions in our file.
    - Plot the resolutions as a 1D histogram AND as a 2D histogram of resolution vs (true) $\eta$ and true $P$.

Some hints - 

- For our efficiency. We need to compare our *thrown* particles of a given type to our *detected* particles of a given type. 
    - When we do our division, we should do this for the same quantity in each case (i.e. compare the true $\eta$ and $P$ values to each other).
    - How can you select the thrown MC Particles of a specific type?
    - How can you select the particles we detected of a specific type?
        - Note, this does not mean we need our *reconstructed* values.
- For our resolution, you might find it helpful to present your result as a percentage.
    - I.e take - (Rec - True)/True)*100 as we did in our example above.
    - Note that again, you will need to be careful as to how you are determining these quantities. You will need to subtract from a reconstructed value it's corresponding MC truth value.

#### Exercise - Major Hint (Reveal if you're very stuck!)

Getting the right arrays here is a bit tricky. We want three different things -

- Our MC particles (truth information), regardless of whether we have a matching track or not. This is just:
    - "MCPartBr['MCParticles.QUANTITY']"[SELECTION_CUTS] - We do **not** need to index this by the SimID
- Our MC particles (truth information) that *do* have a matching reconstructed track, we just need to index these by the SimID:
    - "MCPartBr['MCParticles.QUANTITY'][SimID]" - We can then apply selection criteria
- The Reconstructed particle information for events which correspond to a real MC track, we just need to index these by our RecID:
    - "ReconChPartBr['ReconstructedChargedParticles.QUANTITY'][RecID]"